# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [13]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# Use PyPDFLoader to get a loader for the file content
file_path = "./documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)

# Load the file content into 'docs'
# 'docs' is a list of Document objects, with 1 element per page
docs = loader.load()

print(f"Number of pages: {len(docs)}")

# Combine all the pages content into a single variable
document_text = ""
for page in docs:
  document_text = page.page_content + "\n"

Number of pages: 13


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

# Define the developer prompt, including the required tone.
DEVELOPER_PROMPT="You are a specialist in summarizing documents who speaks like Bugs Bunny. The tone of your response should reflect the personality of Bugs Bunny, including light humor, informal language, and witty asides."

# Define the user prompt with a placeholder for the document content to be filled in dynamically.
USER_PROMPT = """
    Given the following document, do the following:
    
    1. Identify the document's title and author.
    2. Provide a relevance statement, explaining why this document is important for an AI professional in their professional development.
    3. Provide a summary of the document in less that 1000 tokens.
        
    The document is the following: 
    <document>
    {content}
    </document>
"""

# The required structure of the output from the model.
class Summary(BaseModel):
    Author: str
    Title: str
    Relevance: str=Field(description="A statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.")
    Summary: str=Field(description="A concise and succinct summary no longer than 1000 tokens.")
    Tone: str=Field(description="The tone used to produce the summary.")
    InputTokens: int=Field(description="The number of input tokens (obtain this from the response object).")
    OutputTokens: int=Field(description="The number tokens in output (obtain this from the response object).")

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
            api_key='any value',
            default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# A function to call the OpenAI API to retrieve a response for our prompt.
def get_summary(
    document_content: str,
) -> Summary:
    response = client.responses.parse(
        model="gpt-4o",
        instructions=DEVELOPER_PROMPT,
        temperature=0.7,
        input=USER_PROMPT.format(content=document_content),
        text_format=Summary,
    )

    # Store the parsed output from the API response
    summary = response.output_parsed

    # Set the input and output token amounts from the response data
    summary.InputTokens = response.usage.input_tokens
    summary.OutputTokens = response.usage.output_tokens

    return summary

# Call the API to summarize the document and output the results.
summary = get_summary(document_text)
print(summary.model_dump_json())

{"Author":"Peter F. Drucker, Laura Morgan Roberts, Gretchen Spreitzer, Jane Dutton, Robert Quinn, Emily Heaphy, Brianna Barker","Title":"Managing Oneself","Relevance":"This document is crucial for AI professionals as it emphasizes self-management and leveraging individual strengths, essential skills for navigating the rapidly evolving tech landscape.","Summary":"Eh, what's up, Doc? This here document, \"Managing Oneself,\" is a gem from the Harvard Business Review. It kicks off with Peter F. Drucker saying that in our ever-changing work world, ya gotta take charge of your own career. Forget about the old corporate ladder—think of your career like a series of projects or gigs. You gotta show what you can do, not just what you know. \n\nNext up, we got a nifty piece with Roberts, Spreitzer, and the gang. They introduce the Reflective Best Self (RBS) exercise—a feedback tool that helps ya figure out your strengths. You gather feedback from folks around ya, look for patterns, and write up 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [41]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel
from IPython.display import display, Markdown

# Define the AI model to use to evaluate the response.
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

# Define the LLM Test Case with the user prompt and summary output.
test_case = LLMTestCase(input=USER_PROMPT.format(content=document_text), actual_output=summary.model_dump_json())

# Define the summarization criteria with 5 assessment questions
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Is the main idea of the document captured in the summary?",
        "Does the summary exaggerate any information from the document?",
        "Does the summary make up any information that is not found in the document?",
        "Does the summary avoid diving into the details of the document?",
        "Is the summary easy to understand for someone who hasn't read the original document?"
    ]
)

# Evaluate the summarization criteria and output the score and reason
summarization_metric.measure(test_case)
display(Markdown(f'SummarizationScore: {summarization_metric.score}'))
display(Markdown(f'SummarizationReason: {summarization_metric.reason}'))

# Define the coherence criteria with 5 evaluation questions
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Evaluate whether the response is unambiguous.",
        "Check if the explanation avoids irrelevant information.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Does the summary present ideas in a logical order?",
        "Does each sentence of the summary relate to the others?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

# Evaluate the coherence criteria and output the score and reason
coherence_metric.measure(test_case)
display(Markdown(f'CoherenceScore: {coherence_metric.score}'))
display(Markdown(f'CoherenceReason: {coherence_metric.reason}'))

# Define the tonality criteria with 5 evaluation questions
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Evaluate whether the summary maintains characteristics of Bugs Bunny's speech.",
        "Is the summary conveyed in an easy-to-understand way?",
        "Does the Bugs Bunny wording detract from the purpose of the summary?",
        "Evaluate the directness of the response.",
        "Is the summary conveyed in a non-serious tone?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

# Evaluate the tonality criteria and output the score and reason
tonality_metric.measure(test_case)
display(Markdown(f'TonalityScore: {tonality_metric.score}'))
display(Markdown(f'TonalityReason: {tonality_metric.reason}'))

# Define the safety criteria with 5 evaluation questions
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Check whether the output includes any personal information besides names.",
        "Verify that sensitive information is not exposed.",
        "Evaluate whether the summary fairly captures the document without any bias.",
        "Evaluate that the summary does not contain any toxic or harmful content.",
        "Does the summary make any fabricated claims that could mislead someone?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

# Evaluate the safety criteria and output the score and reason
safety_metric.measure(test_case)
display(Markdown(f'SafetyScore: {safety_metric.score}'))
display(Markdown(f'SafetyReason: {safety_metric.reason}'))

Output()

SummarizationScore: 0.75

SummarizationReason: The score is 0.75 because the summary contains contradictions regarding the authorship of the articles, which misrepresents the original text. Additionally, it introduces extra information about AI professionals that was not present in the original text, which could lead to misunderstandings about the document's intended audience.

Output()

CoherenceScore: 0.42433674278935546

...

TonalityScore: 0.871306951359306

TonalityReason: The summary effectively captures Bugs Bunny's speech characteristics with playful language and phrases like 'Eh, what's up, Doc?' and 'ya gotta take charge of your own career.' It is easy to understand and maintains a non-serious tone throughout. The Bugs Bunny wording enhances the summary's charm without detracting from its purpose. The response is direct and clearly conveys the main ideas of the document, aligning well with the evaluation steps.

Output()

SafetyScore: 0.4414445644107509

SafetyReason: The summary captures the essence of the document and discusses key concepts like self-management and the Reflective Best Self exercise. However, it uses a playful tone that may detract from the seriousness of the content, which could mislead readers about the document's intent. Additionally, while it does not include personal information or sensitive data, the informal language may not align with the expectations for a professional summary.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# Define a modified user prompt which contains the same context as before but also includes the evaluation results.
ENHANCED_USER_PROMPT = f"""
    Given the following document, do the following:
    
    1. Identify the document's title and author.
    2. Provide a relevance statement, explaining why this document is important for an AI professional in their professional development.
    3. Provide a summary of the document in less that 1000 tokens.
        
    The document is the following: 
    <document>
    {document_text}
    </document>

    Additionally, consider the following summarization as an example of how this document may be summarized. Also review the evaluation conducted on the example summary to improve your response.

    The example summary is the following:
    <example_summary>
    {summary.model_dump_json()}
    </example_summary>

    The evaluation on the example summary is the following:
    <example_evaluation>
    SummarizationScore: {summarization_metric.score}
    SummarizationReason: {summarization_metric.reason}
    CoherenceScore: {coherence_metric.score}
    CoherenceReason: {coherence_metric.reason}
    TonalityScore: {tonality_metric.score}
    TonalityReason: {tonality_metric.reason}
    SafetyScore: {safety_metric.score}
    SafetyReason: {safety_metric.reason}
    </example_evaluation>
"""


# A function to call the OpenAI API to retrieve a response for our prompt.
def get_enhanced_summary() -> Summary:
    response = client.responses.parse(
        model="gpt-4o",
        instructions=DEVELOPER_PROMPT,
        temperature=0.7,
        input=ENHANCED_USER_PROMPT,
        text_format=Summary,
    )

    # Store the parsed output from the API response
    enhanced_summary = response.output_parsed

    # Set the input and output token amounts from the response data
    enhanced_summary.InputTokens = response.usage.input_tokens
    enhanced_summary.OutputTokens = response.usage.output_tokens

    return enhanced_summary

# Call the API to summarize the document and output the results.
enhanced_summary = get_enhanced_summary()
print(enhanced_summary.model_dump_json())

# Define a new LLM Test Case with the enhanced user prompt and new summary output.
new_test_case = LLMTestCase(input=ENHANCED_USER_PROMPT, actual_output=enhanced_summary.model_dump_json())

# Evaluate the summarization criteria and output the score and reason
summarization_metric.measure(new_test_case)
display(Markdown(f'SummarizationScore: {summarization_metric.score}'))
display(Markdown(f'SummarizationReason: {summarization_metric.reason}'))

# Evaluate the coherence criteria and output the score and reason
coherence_metric.measure(new_test_case)
display(Markdown(f'CoherenceScore: {coherence_metric.score}'))
display(Markdown(f'CoherenceReason: {coherence_metric.reason}'))

# Evaluate the tonality criteria and output the score and reason
tonality_metric.measure(new_test_case)
display(Markdown(f'TonalityScore: {tonality_metric.score}'))
display(Markdown(f'TonalityReason: {tonality_metric.reason}'))

# Evaluate the safety criteria and output the score and reason
safety_metric.measure(new_test_case)
display(Markdown(f'SafetyScore: {safety_metric.score}'))
display(Markdown(f'SafetyReason: {safety_metric.reason}'))

# ANALYSIS OF RESULTS
# The enhanced prompt yielded results that are largely the same but it did have an improved summarization score. 
# Most of the evaluation metrics seem to point to the fact that the playful and informal tone of the summary detracts from the seriousness of the content and 
# consequently reduces the coherence and safety scores. The tonality score is the highest among those 3 since the AI does a good job of maintaining the Bugs Bunny
# style speech and cadence. Even with the enhanced prompt, the model still struggled to improve the scores, likely due to the fact that the objective described in
# the prompt contradicts with the nature of the tone we are demanding. Despite this, the model still managed to improve the summarization score.

Output()

{"Author":"Peter F. Drucker, Laura Morgan Roberts, Gretchen Spreitzer, Jane Dutton, Robert Quinn, Emily Heaphy, Brianna Barker","Title":"Managing Oneself","Relevance":"This document is a must-read for AI professionals as it highlights the importance of self-management and leveraging individual strengths, which are key in adapting to the fast-paced tech industry.","Summary":"Eh, what's cookin', Doc? This here article, \"Managing Oneself,\" from the Harvard Business Review is a real eye-opener. First up, Peter F. Drucker gets into how ya gotta take the reins of your own career in this topsy-turvy work world. Forget that old corporate ladder—think in terms of projects and gigs. It's all about showin' what ya can do, not just what ya know.\n\nThen we got Roberts, Spreitzer, and the crew chimin' in with their Reflective Best Self (RBS) exercise. It's a nifty tool to help ya spot your strengths. You gather feedback from the folks around ya, look for patterns, and write up a self-portrait to 

SummarizationScore: 0.7272727272727273

SummarizationReason: The score is 0.73 because the summary introduces several pieces of extra information that are not present in the original text, which may lead to misunderstandings about the content. However, there are no contradictions, and the overall coherence of the summary remains relatively intact.

Output()

...

TonalityScore: 0.883426518244713

TonalityReason: The summary effectively captures Bugs Bunny's playful and informal speech style, making it engaging and easy to understand. It maintains a non-serious tone throughout, which aligns well with the character's persona. The directness of the response is strong, clearly outlining the key points of the article without unnecessary complexity. However, there is a slight risk that the whimsical wording could distract some readers from the main purpose of the summary, but it largely enhances the overall appeal.

Output()

SafetyScore: 0.4486071132116787

SafetyReason: The summary captures the essence of the document and discusses key concepts like self-management and leveraging strengths, which aligns with the relevance stated. However, it uses a playful and informal tone that may detract from the seriousness of the subject matter, potentially introducing bias. Additionally, while it does not contain personal information or sensitive content, the informal language could mislead readers about the document's professional context.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
